# Solution C1 · Duplicate rows: the OOB error (and the cross-validation) say 100%

**Type**: Break the model

## Objective

Build a noisy dataset in which every row appears **twice** (or almost: with a tiny measurement noise) and show that both the OOB accuracy and a random cross-validation become optimistic, because the copy of an example that is "outside the sample" is often inside. Fix it with
validation **by group** (`GroupKFold`) and with the removal of the duplicates.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_val_score

X, y = make_classification(1000, n_features=10, n_informative=2, n_redundant=0, flip_y=0.4, random_state=3)
X_d, y_d = np.r_[X, X], np.r_[y, y]
group = np.r_[np.arange(1000), np.arange(1000)]
X_q = np.r_[X, X + np.random.default_rng(0).normal(0, 1e-3, X.shape)]
rf = lambda: RandomForestClassifier(200, oob_score=True, random_state=0, n_jobs=-1)

## Commented solution

In [ ]:
clean_oob = rf().fit(X, y).oob_score_
dup_oob = rf().fit(X_d, y_d).oob_score_
near_oob = rf().fit(X_q, y_d).oob_score_
print(f"OOB: clean {clean_oob:.3f} | exact duplicates {dup_oob:.3f} | near-duplicates {near_oob:.3f}")

In [ ]:
random_cv = cross_val_score(rf(), X_d, y_d, cv=StratifiedKFold(5, shuffle=True, random_state=0)).mean()
group_cv = cross_val_score(rf(), X_d, y_d, groups=group, cv=GroupKFold(5)).mean()
oob_inflation = dup_oob - clean_oob
cv_inflation = random_cv - group_cv
print(f"cross-validation: random {random_cv:.3f} | by group {group_cv:.3f} | OOB inflation {oob_inflation:+.3f}, cross-validation {cv_inflation:+.3f}")

In [ ]:
data = pd.DataFrame(np.c_[X_d, y_d]).drop_duplicates().to_numpy()
oob_after_dedup = rf().fit(data[:, :-1], data[:, -1].astype(int)).oob_score_
n_rows_after_near_dedup = len(pd.DataFrame(X_q).drop_duplicates())
print(f"rows after removing the exact duplicates: {len(data)} | OOB: {oob_after_dedup:.3f}")
print(f"near-duplicates: drop_duplicates leaves {n_rows_after_near_dedup} rows out of {len(X_q)} (none removed)")

## Check

In [ ]:
with verify():
    check(0.7 < clean_oob < 0.82, "on the clean dataset the OOB is realistic (0.70-0.82)", "RandomForestClassifier(oob_score=True) on X, y")
    check(dup_oob > 0.97 and near_oob > 0.95, "with exact duplicates the OOB goes over 0.97; with near-duplicates 0.95", "the copy of an 'outside' example is inside the bootstrap")
    check(random_cv > 0.9 and 0.7 < group_cv < 0.82, "random cross-validation on the duplicates goes over 0.9; by group it goes back to 0.70-0.82", "GroupKFold(5) with groups=group")
    check(oob_inflation > 0.2 and cv_inflation > 0.15 and close(oob_inflation, dup_oob - clean_oob), "inflation of over 0.2 for the OOB and over 0.15 for the cross-validation", "differences with the correct estimate")
    check(abs(oob_after_dedup - clean_oob) < 0.02 and n_rows_after_near_dedup == 2000, "after removing the duplicates the OOB goes back to the clean one; the near-duplicates are not removed (2000 rows stay)", "drop_duplicates compares identical values")
done()

## What you learned

- **The OOB assumes independent examples.** If every row has a copy, for a tree that does not have a row in its own bootstrap the copy is there with probability 63%: the tree has already "seen" it and remembers it. Result: OOB accuracy over 0.99 on data whose true generalization is about 0.76.
- **Random cross-validation is optimistic too** (0.94): the validation fold contains copies of training rows. The remedy is `GroupKFold` with a group identifier (customer, patient, source): honest estimate 0.75 (0.748). It is the same problem as M3 C2, inside a model that amplifies it (a forest memorizes).
- **Near-duplicates are worse**: `drop_duplicates` does not see them (the values differ by 1e-3) and the OOB stays inflated. They are found by grouping on identifiers, with rounded hashes or with a nearest-neighbor search. In real data: repeated measurements of the same subject, images of consecutive frames, versions of a document.
- A warning sign: **very high OOB or CV accuracy on a problem you expect to be noisy**, together with a huge gap with a test set collected at another time or from other subjects.

## Common pitfall

Trusting the OOB (or a random K-fold) without knowing whether the rows are independent. Always ask: "are there rows that describe the same object?" If so, validate by group.

## Interview question

> *Click on a question to show its answer.*

<details><summary>When does the out-of-bag error mislead?</summary>

When the examples are not independent: duplicates, repeated measurements of the same subject, consecutive frames, temporal data. The copy of an "out-of-sample" example for a tree is often in its bootstrap, so the tree has already seen it and the OOB error is too low. In those cases I validate by group
(GroupKFold on customer/patient) or by time, and I remove the exact and near duplicates.

</details>